# Notebook 2 — Alur Proyek FERQUEUE (SSD MobileNet V2 FPNLite 320×320)

Notebook Google Colab ini membedah secara transparan seluruh tahapan **Computer Vision yang benar-benar berjalan di proyek FERQUEUE** menggunakan model hasil trainingmu sendiri (`pcv-model-colab-v3-ckpt-35.zip` / `SavedModel`, 3 kelas: `Mobil`, `Truk`, `Bus`, dilatih pada 904 train / 387 test selama 30 epoch).

### Keunggulan Notebook Ini
* **Tidak perlu instalasi TF Object Detection API / protobuf / venv** — cukup pakai TensorFlow bawaan Google Colab untuk memuat `SavedModel` (`saved_model.pb`) dan membedah isi *Feature Maps* langsung dari bobot modelmu sendiri!
* **Memvisualisasikan tahapan otomatis MobileNetV2**:
  0. **Daftar Lengkap & Diagram Arsitektur Network CNN** — Menampilkan *list* seluruh layer CNN (Backbone MobileNetV2 → Neck FPNLite P3–P7 → Weight-Shared SSD Tower & Box/Class Heads), bentuk kernel & output shape, jumlah parameter trainable/non-trainable, serta diagram arsitektur 300 DPI.
  1. **Preprocessing** (Eksternal `processing.py` vs Internal SSD: Resize `320×320`, Normalisasi `[-1, 1]`, & Augmentasi training).
  2. **Ekstraksi Fitur Otomatis** — Mengekstrak **Feature Maps nyata** dari *Low-Level* (`160×160×32`), *Mid-Level* (`80×80×96`), hingga 5 skala **FPNLite (P3–P7)** + **Activation Heatmap Overlay**.
  3. **Deteksi & Post-Processing (`postprocessing.py`)** — Membandingkan **Raw Candidate Boxes (12.804 anchor)** vs **Confidence Filtering (`>= 0.5`)** vs **Hasil Akhir NMS (`IoU <= 0.45`)**.
  4. **Segmentasi Pasca-Deteksi (`processing.py`)** — Menerapkan **GrabCut + Morfologi** pada ROI bounding box hasil prediksi (jujur terhadap arsitektur: SSD memprediksi bbox dari gambar utuh, GrabCut memisahkan *foreground* untuk review/analisis bentuk).
  5. *(Opsional)* **Kalibrasi Koin 50 Yen (`side_measurement.py`)** — Segmentasi Otsu + Contour + Fit Ellipse untuk estimasi panjang & tinggi kendaraan (cm).

## Cell 1 — Setup Library & Folder Output Presentasi

In [ ]:
import json
import math
import shutil
import zipfile
from pathlib import Path
from time import perf_counter

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image, ImageOps
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2

try:
    from google.colab import drive, files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

try:
    display  # type: ignore[name-defined]
except NameError:
    display = print

OUT_DIR = Path("/content/outputs_presentasi/nb2_ferqueue_ssd") if IN_COLAB else Path("./outputs_presentasi/nb2_ferqueue_ssd")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_WORK_DIR = Path("/content/ferqueue_model_viz") if IN_COLAB else Path("./outputs_presentasi/_model_cache")
MODEL_WORK_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.titleweight"] = "bold"


def save_fig(fig, filename: str):
    path = OUT_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"Tersimpan: {path}")
    return path


print(f"TensorFlow={tf.__version__} | OpenCV={cv2.__version__} | GPU={len(tf.config.list_physical_devices('GPU')) > 0}")

## Cell 2 — Muat Model FERQUEUE (`pcv-model-colab-v3-ckpt-35.zip`) & Gambar Uji

Pilih sumber model lewat `MODEL_SOURCE`:
* `"upload_zip"` *(paling praktis di Colab)*: unggah file `outputs/pcv-model-colab-v3-ckpt-35.zip` (~10 MB) langsung dari laptop.
* `"drive"`: baca ZIP atau folder `saved_model` dari Google Drive (ubah `DRIVE_MODEL_PATH`).
* `"local"`: fallback jika dijalankan lokal di root repo FERQUEUE.

In [ ]:
MODEL_SOURCE = "upload_zip"   # Pilihan: "upload_zip" | "drive" | "local"
DRIVE_MODEL_PATH = "/content/drive/MyDrive/ferqueue/pcv-coco-colab-v3/pcv-model-colab-v3-ckpt-35.zip"
LOCAL_MODEL_PATH = "models/mobilenet/runs/pcv-receiver-v3-unfrozen/exports/imported-ckpt-35"

IMAGE_SOURCE = "upload"       # Pilihan: "upload" | "drive" | "local"
DRIVE_IMAGE_DIR = "/content/drive/MyDrive/ferqueue/sample_images"
LOCAL_IMAGE_DIR = "outputs/test-samples"
ACTIVE_INDEX = 0              # Indeks gambar utama yang dibedah di Cell 3-7

DEFAULT_CLASSES = ["Mobil", "Truk", "Bus"]
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}


def resolve_saved_model(src_mode: str):
    extract_dir = MODEL_WORK_DIR / "extracted"
    meta = {
        "classes": DEFAULT_CLASSES,
        "model": "ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8",
        "checkpoint": "ckpt-35",
        "preprocessing": {"denoise_kernel": 0, "clahe_clip_limit": 0, "clahe_grid_size": 8, "gamma": 1.0},
    }

    if src_mode == "upload_zip" and IN_COLAB:
        print("Silakan unggah file ZIP model (misal: pcv-model-colab-v3-ckpt-35.zip)...")
        up = colab_files.upload()
        zip_name, content = next(iter(up.items()))
        zip_file = MODEL_WORK_DIR / Path(zip_name).name
        zip_file.write_bytes(content)
        target_path = zip_file
    elif src_mode == "drive" and IN_COLAB:
        drive.mount("/content/drive")
        target_path = Path(DRIVE_MODEL_PATH)
    else:
        candidates = [
            Path(LOCAL_MODEL_PATH),
            Path("models/mobilenet/runs/pcv-receiver-v4/exports/imported-ckpt-42"),
            Path("models/mobilenet/runs/pcv-receiver-isnet/exports/imported-ckpt-7"),
            Path("outputs/pcv-model-colab-v3-ckpt-35.zip"),
        ]
        target_path = next((p for p in candidates if p.exists()), Path(LOCAL_MODEL_PATH))

    if target_path.is_file() and target_path.suffix.lower() == ".zip":
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(target_path) as zf:
            zf.extractall(extract_dir)
        if (extract_dir / "model.json").is_file():
            meta.update(json.loads((extract_dir / "model.json").read_text(encoding="utf-8")))
        saved_dir = extract_dir / "saved_model"
    elif target_path.is_dir():
        saved_dir = target_path / "saved_model" if (target_path / "saved_model/saved_model.pb").is_file() else target_path
        export_json = target_path.parent.parent / "export.json"
        if export_json.is_file():
            meta.update(json.loads(export_json.read_text(encoding="utf-8")))
    else:
        raise FileNotFoundError(f"Model tidak ditemukan di: {target_path}")

    if not (saved_dir / "saved_model.pb").is_file():
        raise FileNotFoundError(f"saved_model.pb tidak ditemukan di dalam: {saved_dir}")
    return saved_dir, meta


saved_model_dir, model_meta = resolve_saved_model(MODEL_SOURCE)
CLASSES = model_meta["classes"]
MODEL_PREPROCESSING = model_meta.get("preprocessing", {})

print(f"Memuat SavedModel dari: {saved_model_dir} ...")
t0 = perf_counter()
loaded_tf_model = tf.saved_model.load(str(saved_model_dir))
detector_fn = loaded_tf_model.signatures["serving_default"]
print(f"Model siap dalam {perf_counter() - t0:.2f} detik!")
print(f"Arsitektur : {model_meta['model']} | Checkpoint: {model_meta['checkpoint']}")
print(f"Kelas      : {CLASSES}")
print(f"Preprocess : {MODEL_PREPROCESSING}")

# Muat gambar uji
UPLOAD_DIR = OUT_DIR / "_inputs"
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
image_paths = []
if IMAGE_SOURCE == "upload" and IN_COLAB:
    print("\nSilakan unggah 1 atau lebih gambar kendaraan uji (JPG/PNG)...")
    uploaded_imgs = colab_files.upload()
    for fname, content in uploaded_imgs.items():
        dst = UPLOAD_DIR / Path(fname).name
        dst.write_bytes(content)
        if dst.suffix.lower() in IMAGE_EXTS:
            image_paths.append(dst)
elif IMAGE_SOURCE == "drive" and IN_COLAB:
    drive.mount("/content/drive")
    image_paths = sorted(p for p in Path(DRIVE_IMAGE_DIR).iterdir() if p.suffix.lower() in IMAGE_EXTS)
else:
    for candidate_dir in (Path(LOCAL_IMAGE_DIR), Path("data/processed/pcv-notebook-samples"), Path("data/inference")):
        if candidate_dir.is_dir():
            image_paths = sorted(p for p in candidate_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)
            if image_paths:
                break

if not image_paths:
    raise FileNotFoundError("Belum ada gambar uji yang dimuat.")


def load_pil_rgb(path: Path) -> Image.Image:
    with Image.open(path) as img:
        return ImageOps.exif_transpose(img).convert("RGB")


loaded_images = [(p.name, load_pil_rgb(p)) for p in image_paths]
ACTIVE_INDEX = min(ACTIVE_INDEX, len(loaded_images) - 1)
active_name, active_pil = loaded_images[ACTIVE_INDEX]
img_rgb = np.asarray(active_pil, dtype=np.uint8)
print(f"\nGambar aktif [{ACTIVE_INDEX}]: {active_name} ({img_rgb.shape[1]}x{img_rgb.shape[0]} px)")

## Cell 2B — Daftar Lengkap & Diagram Arsitektur Network CNN (SSD MobileNet V2 FPNLite 320×320)

Cell ini membedah dan menampilkan **daftar lengkap (*list*) network CNN** langsung dari `SavedModel` yang baru dimuat di Cell 2 (tanpa perlu instalasi TF Object Detection API):
* **Backbone — MobileNetV2**: Stem `Conv1` + 17 blok *Inverted Residual Bottleneck* (`DepthwiseConv2D` + `Conv2D` 1×1 + `BatchNormalization` + `ReLU6`).
* **Neck — FPNLite (P3–P7)**: Proyeksi lateral 1×1, *top-down upsampling*, *smoothing* `SeparableConv2D` (P3–P5), serta 2 tahap *bottom-up stride-2* (P6–P7).
* **Head — Weight-Shared SSD Box & Class Predictor**: *Prediction Tower* (4× `SeparableConv2D` 3×3 @128 ch yang bobotnya dibagi ke 5 skala FPN + `BatchNorm` terpisah per skala) dan *dual head* (`BoxPredictor` & `ClassPredictor`) pada **12.804 anchors** untuk 3 kelas (`Mobil`, `Truk`, `Bus`) + 1 *background*.

Output yang dihasilkan:
1. Ringkasan global parameter (*trainable* vs *non-trainable*) & tabel per komponen/tipe layer.
2. **Daftar lengkap seluruh layer network CNN** (ditampilkan utuh tanpa terpotong + ekspor ke `00_cnn_network_layers.csv` & `00_cnn_network_summary.txt`).
3. **Diagram arsitektur network CNN + grafik distribusi parameter (300 DPI)** (`00_arsitektur_network_cnn.png`).

In [ ]:
from collections import OrderedDict
from matplotlib.patches import FancyBboxPatch

print("Menganalisis struktur & daftar layer network CNN dari SavedModel...")
frozen_fn = convert_variables_to_constants_v2(detector_fn, lower_control_flow=False)

# Petakan output shape 4D dari seluruh operasi di frozen graph
_shape_by_op = {}
for _op in frozen_fn.graph.get_operations():
    if not _op.outputs:
        continue
    _shp = _op.outputs[0].shape
    if _shp.rank == 4:
        _shape_by_op[_op.name] = tuple(int(d) if d is not None else None for d in _shp.as_list())
_op_names = list(_shape_by_op.keys())


def _classify_cnn_component(layer_name: str) -> str:
    low = layer_name.lower()
    if "predictiontower" in low or "box_encodings" in low:
        return "Head (SSD Shared Tower)"
    if any(tok in low for tok in ("boxhead", "classhead", "classpredictor")) or low.endswith("boxpredictor"):
        return "Head (SSD Box & Class Predictor)"
    if "bottom_up_conv2d" in low:
        return "Neck (FPNLite Coarse P6-P7)"
    if any(tok in low for tok in ("projection_", "smoothing_", "nearest_neighbor_upsampling", "top_down")):
        return "Neck (FPNLite Top-Down P3-P5)"
    return "Backbone (MobileNetV2)"


def _infer_layer_type(layer_name: str, var_kinds) -> str:
    low = layer_name.lower()
    kinds = set(var_kinds)
    if "depthwise_kernel" in kinds and "pointwise_kernel" in kinds:
        return "SeparableConv2D"
    if "depthwise_kernel" in kinds:
        return "DepthwiseConv2D"
    if "kernel" in kinds:
        return "Conv2D"
    if "batchnorm" in low or low.endswith("_bn") or low.startswith("bn_") or {"gamma", "beta"} <= kinds:
        return "BatchNormalization"
    return "Layer"


def _find_output_shape(layer_name: str) -> str:
    low = layer_name.lower()
    if "predictiontower/conv2d_" in low and "batchnorm" not in low:
        return "Shared (P3..P7, 128 ch)"
    if "boxhead" in low or low.endswith("boxpredictor"):
        return "Shared (P3..P7, 24 ch) -> (1, 12804, 4)"
    if "classhead" in low or "classpredictor" in low:
        return f"Shared (P3..P7, {(len(CLASSES) + 1) * 6} ch) -> (1, 12804, {len(CLASSES) + 1})"
    needle = "/" + layer_name + "/"
    matches = [_shape_by_op[n] for n in _op_names if needle in ("/" + n + "/")]
    for shp in reversed(matches):
        if shp[1] is not None and shp[2] is not None:
            return "(" + ", ".join("?" if d is None else str(d) for d in shp) + ")"
    if matches:
        return "(" + ", ".join("?" if d is None else str(d) for d in matches[-1]) + ")"
    return "-"


# Kelompokkan seluruh variabel bobot SavedModel menjadi daftar layer CNN
_grouped = OrderedDict()
for _v in detector_fn.variables:
    _clean = _v.name.split(":")[0]
    _parts = _clean.split("/")
    _lname = "/".join(_parts[:-1]) if len(_parts) > 1 else _clean
    _pkind = _parts[-1]
    _entry = _grouped.setdefault(
        _lname,
        {"var_kinds": [], "weight_shapes": [], "params": 0, "trainable": 0, "non_trainable": 0},
    )
    _n = int(tf.size(_v))
    _entry["var_kinds"].append(_pkind)
    _entry["params"] += _n
    if _v.trainable:
        _entry["trainable"] += _n
    else:
        _entry["non_trainable"] += _n
    if _pkind in ("kernel", "depthwise_kernel", "pointwise_kernel", "weights"):
        _entry["weight_shapes"].append(f"{_pkind}={tuple(int(d) for d in _v.shape)}")

cnn_layer_rows = []
for _idx, (_lname, _info) in enumerate(_grouped.items(), start=1):
    cnn_layer_rows.append({
        "No": _idx,
        "Komponen": _classify_cnn_component(_lname),
        "Nama_Layer": _lname,
        "Tipe_Layer": _infer_layer_type(_lname, _info["var_kinds"]),
        "Shape_Kernel": ", ".join(_info["weight_shapes"]) if _info["weight_shapes"] else "-",
        "Output_Shape": _find_output_shape(_lname),
        "Param": _info["params"],
        "Trainable_Param": _info["trainable"],
        "Non_Trainable_Param": _info["non_trainable"],
    })

df_cnn_layers = pd.DataFrame(cnn_layer_rows)
total_params = int(df_cnn_layers["Param"].sum())
total_trainable = int(df_cnn_layers["Trainable_Param"].sum())
total_non_trainable = int(df_cnn_layers["Non_Trainable_Param"].sum())
size_mb_fp32 = total_params * 4 / (1024 * 1024)

# Hitung juga operasi non-parametrik utama (aktivasi ReLU6, residual Add) di graph
_relu6_ops = sum(1 for op in frozen_fn.graph.get_operations() if op.type == "Relu6")
_add_ops = sum(1 for op in frozen_fn.graph.get_operations() if op.type in ("Add", "AddV2") and ("_add/" in op.name or "add" in op.name.lower()) and "block_" in op.name)

# 1. Ringkasan per komponen
COMPONENT_ORDER = [
    "Backbone (MobileNetV2)",
    "Neck (FPNLite Top-Down P3-P5)",
    "Neck (FPNLite Coarse P6-P7)",
    "Head (SSD Shared Tower)",
    "Head (SSD Box & Class Predictor)",
]
df_comp_summary = (
    df_cnn_layers.groupby("Komponen", as_index=False)
    .agg(
        Jumlah_Layer=("No", "count"),
        Total_Param=("Param", "sum"),
        Trainable=("Trainable_Param", "sum"),
        Non_Trainable=("Non_Trainable_Param", "sum"),
    )
)
df_comp_summary["__ord"] = df_comp_summary["Komponen"].apply(lambda c: COMPONENT_ORDER.index(c) if c in COMPONENT_ORDER else 99)
df_comp_summary = df_comp_summary.sort_values("__ord").drop(columns=["__ord"]).reset_index(drop=True)
df_comp_summary["Porsi_%"] = (df_comp_summary["Total_Param"] / total_params * 100.0).round(2)

# 2. Ringkasan per tipe layer berparameter
df_type_summary = (
    df_cnn_layers.groupby("Tipe_Layer", as_index=False)
    .agg(Jumlah_Layer=("No", "count"), Total_Param=("Param", "sum"))
    .sort_values(["Total_Param", "Jumlah_Layer"], ascending=[False, False])
    .reset_index(drop=True)
)

print("=" * 108)
print(f"RINGKASAN NETWORK CNN — {model_meta['model']} ({model_meta['checkpoint']})")
print("=" * 108)
print("Input Shape      : (1, 320, 320, 3) [RGB, normalisasi internal [-1, 1]]")
print(f"Kelas Output     : {CLASSES} ({len(CLASSES)} kelas + 1 background)")
print(f"Layer Berbobot   : {len(df_cnn_layers)} layer ({len(detector_fn.variables)} tensor bobot; + {_relu6_ops} op ReLU6 & {_add_ops} residual Add)")
print(f"Total Parameter  : {total_params:,} (Trainable: {total_trainable:,} | Non-Trainable: {total_non_trainable:,})")
print(f"Ukuran Bobot     : ~{size_mb_fp32:.2f} MB (float32)")
print("Total Anchor SSD : 12,804 anchors (P3 40x40 + P4 20x20 + P5 10x10 + P6 5x5 + P7 3x3, @6 anchors/sel)")
print("=" * 108)

print("\n[A] Ringkasan per Komponen Network CNN:")
display(df_comp_summary)

print("\n[B] Ringkasan per Tipe Layer Berparameter:")
display(df_type_summary)

# Simpan CSV & teks summary lengkap ke OUT_DIR
csv_net_path = OUT_DIR / "00_cnn_network_layers.csv"
df_cnn_layers.to_csv(csv_net_path, index=False)

txt_lines = [
    "=" * 140,
    f"DAFTAR LENGKAP LAYER NETWORK CNN — {model_meta['model']} ({model_meta['checkpoint']})",
    "=" * 140,
    f"Input: (1, 320, 320, 3) | Kelas: {', '.join(CLASSES)} | Layer Berbobot: {len(df_cnn_layers)} | Total Param: {total_params:,} (Trainable: {total_trainable:,} | Non-Trainable: {total_non_trainable:,})",
    "",
    f"{'No':>3}  {'Komponen':<33} {'Nama Layer':<46} {'Tipe':<18} {'Output Shape':<24} {'Param #':>10}",
    "-" * 140,
]
for _, r in df_cnn_layers.iterrows():
    lname_disp = r["Nama_Layer"] if len(r["Nama_Layer"]) <= 46 else "..." + r["Nama_Layer"][-43:]
    txt_lines.append(
        f"{int(r['No']):>3}  {r['Komponen']:<33} {lname_disp:<46} {r['Tipe_Layer']:<18} {r['Output_Shape']:<24} {int(r['Param']):>10,}"
    )
txt_net_path = OUT_DIR / "00_cnn_network_summary.txt"
txt_net_path.write_text("\n".join(txt_lines) + "\n", encoding="utf-8")
print(f"\nTersimpan: {csv_net_path}")
print(f"Tersimpan: {txt_net_path}")

print(f"\n[C] Daftar Lengkap Seluruh {len(df_cnn_layers)} Layer Network CNN:")
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 72, "display.width", 180):
    display(df_cnn_layers[["No", "Komponen", "Nama_Layer", "Tipe_Layer", "Shape_Kernel", "Output_Shape", "Param"]])

# 3. Diagram Arsitektur Network CNN + Distribusi Parameter (300 DPI)
fig_arch = plt.figure(figsize=(18, 11.2))
fig_arch.patch.set_facecolor("white")
gs_arch = fig_arch.add_gridspec(2, 2, height_ratios=[1.42, 1.0], hspace=0.28, wspace=0.22)

ax_diag = fig_arch.add_subplot(gs_arch[0, :])
ax_diag.set_xlim(0, 100)
ax_diag.set_ylim(0, 44)
ax_diag.axis("off")

ax_diag.text(
    50, 42.2,
    "Arsitektur Network CNN — SSD MobileNet V2 FPNLite 320×320 (FERQUEUE)",
    ha="center", va="center", fontsize=15, fontweight="bold", color="#0f172a",
)
ax_diag.text(
    50, 39.5,
    f"Layer Berbobot: {len(df_cnn_layers)} Layer  |  {total_params:,} Parameter "
    f"(Trainable: {total_trainable:,} • Non-Trainable: {total_non_trainable:,})  |  "
    f"~{size_mb_fp32:.2f} MB (FP32)  |  12,804 Anchors  |  Kelas: {', '.join(CLASSES)}",
    ha="center", va="center", fontsize=9.8, color="#334155",
)


def _draw_box(ax, x, y, w, h, title, body, fc, ec, ts=8.2, bs=7.3, toff=0.9, boff=1.25):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.35,rounding_size=0.9", facecolor=fc, edgecolor=ec, linewidth=1.7))
    ax.text(x + w / 2, y + h - toff, title, ha="center", va="top", fontsize=ts, fontweight="bold", color="#0f172a")
    ax.text(x + w / 2, y + h / 2 - boff, body, ha="center", va="center", fontsize=bs, color="#1e293b", linespacing=1.32)


def _draw_arrow(ax, x1, y1, x2, y2, color="#475569"):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="-|>,head_length=0.55,head_width=0.32", color=color, lw=1.6))


_cm = {r["Komponen"]: r for _, r in df_comp_summary.iterrows()}
_bb = _cm.get("Backbone (MobileNetV2)", {"Jumlah_Layer": 0, "Total_Param": 0})
_fp1 = _cm.get("Neck (FPNLite Top-Down P3-P5)", {"Jumlah_Layer": 0, "Total_Param": 0})
_fp2 = _cm.get("Neck (FPNLite Coarse P6-P7)", {"Jumlah_Layer": 0, "Total_Param": 0})
_tw = _cm.get("Head (SSD Shared Tower)", {"Jumlah_Layer": 0, "Total_Param": 0})
_hd = _cm.get("Head (SSD Box & Class Predictor)", {"Jumlah_Layer": 0, "Total_Param": 0})

for gx, gy, gw, gh, gtitle, gfc, gec in [
    (12.6, 2.2, 24.4, 34.8, f"1. BACKBONE: MobileNetV2 ({int(_bb['Jumlah_Layer'])} Lyr | {int(_bb['Total_Param']):,} P)", "#eff6ff", "#2563eb"),
    (39.4, 2.2, 21.6, 34.8, f"2. NECK: FPNLite P3–P7 ({int(_fp1['Jumlah_Layer'] + _fp2['Jumlah_Layer'])} Lyr | {int(_fp1['Total_Param'] + _fp2['Total_Param']):,} P)", "#f0fdf4", "#16a34a"),
    (63.4, 2.2, 23.0, 34.8, f"3. HEAD: Shared SSD ({int(_tw['Jumlah_Layer'] + _hd['Jumlah_Layer'])} Lyr | {int(_tw['Total_Param'] + _hd['Total_Param']):,} P)", "#fff7ed", "#ea580c"),
]:
    ax_diag.add_patch(FancyBboxPatch((gx, gy), gw, gh, boxstyle="round,pad=0.30,rounding_size=1.1", facecolor=gfc, edgecolor=gec, linewidth=1.4, linestyle="--", alpha=0.75))
    ax_diag.text(gx + gw / 2, gy + gh - 0.8, gtitle, ha="center", va="top", fontsize=7.8, fontweight="bold", color=gec)

_draw_box(ax_diag, 0.6, 13.5, 9.8, 13.0, "INPUT CITRA", "RGB Frame\nResize 320×320×3\nNorm [-1, 1]\n(Fixed Shape)", "#f1f5f9", "#475569", ts=9.0, bs=7.8, toff=1.4, boff=1.4)
_draw_arrow(ax_diag, 10.8, 20.0, 13.4, 20.0)

_bb_stages = [
    (13.8, 28.4, 22.0, 5.1, "Stem: Conv1 3×3 (s=2) + BN + ReLU6", "320×320×3 → 160×160×32 | Block 0 → 160×160×16"),
    (13.8, 22.1, 22.0, 5.1, "Stage 2: Inverted Residual Block 1–2", "160×160×16 → 80×80×24 (Depthwise + Pointwise)"),
    (13.8, 15.8, 22.0, 5.1, "Stage 3: Inverted Residual Block 3–5 [C3]", "80×80×24 → 40×40×32 (Fitur detail / objek kecil)"),
    (13.8, 9.5, 22.0, 5.1, "Stage 4: Inverted Residual Block 6–12 [C4]", "40×40×32 → 20×20×96 (Fitur bodi kendaraan sedang)"),
    (13.8, 3.2, 22.0, 5.1, "Stage 5: Block 13–16 + Conv_1 1×1 [C5]", "20×20×96 → 10×10×1280 (Semantik global tinggi)"),
]
for i_s, (bx, by, bw, bh, bt, bd) in enumerate(_bb_stages):
    _draw_box(ax_diag, bx, by, bw, bh, bt, bd, "#dbeafe", "#1d4ed8", ts=8.0, bs=7.2, toff=0.85, boff=1.15)
    if i_s < len(_bb_stages) - 1:
        _draw_arrow(ax_diag, bx + bw / 2, by - 0.1, bx + bw / 2, _bb_stages[i_s + 1][1] + _bb_stages[i_s + 1][3] + 0.15, "#1d4ed8")

for fx, fy, fw, fh, ft, fd in [
    (40.7, 28.4, 19.0, 5.1, "FPN P3 — 40×40×128", "Proj 1×1 + Upsample + SepConv 3×3\n9.600 anchors (kendaraan kecil)"),
    (40.7, 22.1, 19.0, 5.1, "FPN P4 — 20×20×128", "Proj 1×1 + Upsample + SepConv 3×3\n2.400 anchors (sedang)"),
    (40.7, 15.8, 19.0, 5.1, "FPN P5 — 10×10×128", "Proj 1×1 dari C5 (10×10×1280)\n600 anchors (sedang-besar)"),
    (40.7, 9.5, 19.0, 5.1, "FPN P6 — 5×5×128", "Bottom-Up SepConv 3×3 (s=2)\n150 anchors (besar)"),
    (40.7, 3.2, 19.0, 5.1, "FPN P7 — 3×3×128", "Bottom-Up SepConv 3×3 (s=2)\n54 anchors (sangat besar)"),
]:
    _draw_box(ax_diag, fx, fy, fw, fh, ft, fd, "#dcfce7", "#15803d", ts=8.1, bs=7.1, toff=0.55, boff=1.05)

_draw_arrow(ax_diag, 36.2, 18.4, 40.3, 30.9, "#15803d")
_draw_arrow(ax_diag, 36.2, 12.1, 40.3, 24.6, "#15803d")
_draw_arrow(ax_diag, 36.2, 5.8, 40.3, 18.4, "#15803d")
_draw_arrow(ax_diag, 50.2, 21.1, 50.2, 21.95, "#15803d")
_draw_arrow(ax_diag, 50.2, 27.4, 50.2, 28.25, "#15803d")
_draw_arrow(ax_diag, 50.2, 15.6, 50.2, 14.8, "#15803d")
_draw_arrow(ax_diag, 50.2, 9.3, 50.2, 8.5, "#15803d")

_draw_box(ax_diag, 64.8, 22.8, 20.2, 10.6, "Shared Prediction Tower", "4× SeparableConv2D (3×3, 128 ch)\n+ BatchNorm per skala (P3–P7)\n+ Aktivasi ReLU6\nBobot Conv dibagi ke semua skala", "#ffedd5", "#c2410c", ts=8.5, bs=7.5, toff=1.1, boff=1.3)
_draw_box(ax_diag, 64.6, 3.6, 9.7, 15.2, "Box Head", "SepConv 3×3\n24 channel\n(6 anchor ×\n4 offset)\n\nOutput:\n(1, 12804, 4)", "#fef3c7", "#b45309", ts=8.3, bs=7.3, toff=1.1, boff=1.3)
_draw_box(ax_diag, 75.5, 3.6, 9.7, 15.2, "Class Head", f"SepConv 3×3\n{(len(CLASSES) + 1) * 6} channel\n(6 anchor ×\n{len(CLASSES)} cls + bg)\n\nOutput:\n(1, 12804, {len(CLASSES) + 1})", "#fee2e2", "#b91c1c", ts=8.3, bs=7.3, toff=1.1, boff=1.3)
_draw_box(ax_diag, 88.2, 10.5, 11.2, 19.0, "POST-PROCESS", f"12.804 Anchors\n↓\nScore Filter\n(>= thr)\n↓\nNMS (IoU)\n↓\nBBox + Kelas\n({'/'.join(CLASSES)})", "#f3e8ff", "#7e22ce", ts=8.5, bs=7.6, toff=1.4, boff=1.4)

_draw_arrow(ax_diag, 60.1, 28.0, 64.4, 28.0, "#c2410c")
_draw_arrow(ax_diag, 71.5, 22.4, 69.5, 19.2, "#b45309")
_draw_arrow(ax_diag, 78.3, 22.4, 80.3, 19.2, "#b91c1c")
_draw_arrow(ax_diag, 85.6, 13.5, 87.8, 17.8, "#7e22ce")
ax_diag.plot([69.5, 69.5, 87.1, 87.1], [3.2, 1.0, 1.0, 14.0], color="#b45309", lw=1.6, solid_capstyle="round")
_draw_arrow(ax_diag, 86.9, 14.0, 87.8, 14.0, "#b45309")

# Grafik bawah kiri: Distribusi Parameter per Komponen
ax_c = fig_arch.add_subplot(gs_arch[1, 0])
_y_c = np.arange(len(df_comp_summary))
_colors_c = ["#2563eb", "#16a34a", "#22c55e", "#ea580c", "#dc2626"][:len(df_comp_summary)]
_bars_c = ax_c.barh(_y_c, df_comp_summary["Total_Param"], color=_colors_c, edgecolor="#0f172a", linewidth=0.6, alpha=0.88)
ax_c.set_yticks(_y_c)
ax_c.set_yticklabels([c.replace(" (", "\n(") for c in df_comp_summary["Komponen"]], fontsize=8.5)
ax_c.invert_yaxis()
ax_c.set_xlabel("Jumlah Parameter", fontsize=9.5, fontweight="bold")
ax_c.set_title("Distribusi Parameter per Komponen CNN", fontsize=11, fontweight="bold", pad=10)
ax_c.grid(axis="x", linestyle="--", alpha=0.35)
ax_c.set_xlim(0, df_comp_summary["Total_Param"].max() * 1.58)
for _b, (_, _r) in zip(_bars_c, df_comp_summary.iterrows()):
    ax_c.text(_b.get_width() + df_comp_summary["Total_Param"].max() * 0.02, _b.get_y() + _b.get_height() / 2,
              f"{int(_r['Total_Param']):,} ({_r['Porsi_%']:.1f}% | {int(_r['Jumlah_Layer'])} lyr)",
              va="center", ha="left", fontsize=8.1, fontweight="bold", color="#1e293b")

# Grafik bawah kanan: Komposisi Tipe Layer Berbobot
ax_t = fig_arch.add_subplot(gs_arch[1, 1])
_y_t = np.arange(len(df_type_summary))
_bars_t = ax_t.barh(_y_t, df_type_summary["Jumlah_Layer"], color="#0ea5e9", edgecolor="#0f172a", linewidth=0.6, alpha=0.85)
ax_t.set_yticks(_y_t)
ax_t.set_yticklabels(df_type_summary["Tipe_Layer"], fontsize=8.8)
ax_t.invert_yaxis()
ax_t.set_xlabel("Jumlah Layer Berparameter", fontsize=9.5, fontweight="bold")
ax_t.set_title("Komposisi Tipe Layer Berparameter dalam Network CNN", fontsize=11, fontweight="bold", pad=10)
ax_t.grid(axis="x", linestyle="--", alpha=0.35)
ax_t.set_xlim(0, df_type_summary["Jumlah_Layer"].max() * 1.55)
for _b, (_, _r) in zip(_bars_t, df_type_summary.iterrows()):
    ax_t.text(_b.get_width() + df_type_summary["Jumlah_Layer"].max() * 0.02, _b.get_y() + _b.get_height() / 2,
              f"{int(_r['Jumlah_Layer'])} layer ({int(_r['Total_Param']):,} param)",
              va="center", ha="left", fontsize=8.2, color="#1e293b")

save_fig(fig_arch, "00_arsitektur_network_cnn.png")
plt.show()

## Cell 3 — Tahap 1: Preprocessing Proyek FERQUEUE (Eksternal & Internal SSD)

Pada proyek FERQUEUE, preprocessing terbagi menjadi **2 lapisan**:
1. **Preprocessing Eksternal (`ferqueue_cv/processing.py`)**:
   - EXIF orientation fix + konversi RGB (`uint8`, `0–255`).
   - Tersedia filter *geometry-preserving* opsional (Gaussian Blur, CLAHE pada ruang warna LAB, Gamma LUT). Pada model v3 final (`receiver-v3.json`), ketiganya diset **baseline (`0 / 0 / 1.0`)** agar tekstur asli kendaraan tetap utuh.
2. **Preprocessing Internal SSD MobileNet V2 (`pipeline.config`)**:
   - `fixed_shape_resizer`: Resize ke **`320×320` piksel**.
   - **Normalisasi piksel** ke rentang **`[-1.0, +1.0]`** dengan rumus $(x / 127.5) - 1.0$.
   - Saat *training* (`coco-v3-colab.json`), juga diterapkan augmentasi `random_horizontal_flip` dan `random_adjust_brightness (max_delta=0.1)`.

In [ ]:
def preprocess_image_ferqueue(image: Image.Image, settings=None) -> Image.Image:
    """Persis mengikuti apps/computer-vision/src/ferqueue_cv/processing.py."""
    settings = settings or {}
    image = image.convert("RGB")
    kernel = settings.get("denoise_kernel", 0)
    clip = settings.get("clahe_clip_limit", 0)
    gamma = settings.get("gamma", 1.0)
    if not kernel and not clip and gamma == 1.0:
        return image
    pixels = np.asarray(image).copy()
    if kernel:
        pixels = cv2.GaussianBlur(pixels, (kernel, kernel), 0)
    if clip:
        lab = cv2.cvtColor(pixels, cv2.COLOR_RGB2LAB)
        grid = settings.get("clahe_grid_size", 8)
        lab[:, :, 0] = cv2.createCLAHE(clipLimit=clip, tileGridSize=(grid, grid)).apply(lab[:, :, 0])
        pixels = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    if gamma != 1.0:
        table = np.round(255 * (np.arange(256) / 255) ** gamma).astype("uint8")
        pixels = cv2.LUT(pixels, table)
    return Image.fromarray(pixels)


# Terapkan preprocessing model aktif (v3 baseline: RGB asli)
proc_pil = preprocess_image_ferqueue(active_pil, MODEL_PREPROCESSING)
proc_rgb = np.asarray(proc_pil, dtype=np.uint8)

# Simulasi internal SSD MobileNetV2: Resize 320x320 & Normalisasi [-1, 1]
resized_320 = cv2.resize(proc_rgb, (320, 320), interpolation=cv2.INTER_LINEAR)
norm_m1_p1 = (resized_320.astype(np.float32) / 127.5) - 1.0

# Simulasi augmentasi training (horizontal flip + brightness +0.1)
aug_flip = cv2.flip(resized_320, 1)
aug_bright = np.clip(resized_320.astype(np.float32) + 0.1 * 255.0, 0, 255).astype(np.uint8)

fig = plt.figure(figsize=(16, 7.6))
gs = fig.add_gridspec(2, 4, height_ratios=[1.3, 1.0], hspace=0.35, wspace=0.2)

panels = [
    (f"1. Input RGB Asli\n({img_rgb.shape[1]}x{img_rgb.shape[0]}, [0..255])", img_rgb),
    ("2. Internal SSD: Resize\n(320x320, fixed_shape_resizer)", resized_320),
    ("3. Augmentasi Training\n(Random Horizontal Flip)", aug_flip),
    ("4. Augmentasi Training\n(Brightness Delta ±0.1)", aug_bright),
]
for i, (title, im) in enumerate(panels):
    ax = fig.add_subplot(gs[0, i])
    ax.imshow(im)
    ax.set_title(title, fontsize=10)
    ax.axis("off")

ax_h1 = fig.add_subplot(gs[1, 0:2])
ax_h1.hist(resized_320.ravel(), bins=64, range=(0, 256), color="#4c72b0", alpha=0.85)
ax_h1.set_title("Distribusi Piksel Sebelum Normalisasi (uint8: 0 s/d 255)")
ax_h1.set_xlabel("Nilai Piksel")
ax_h1.set_ylabel("Frekuensi")
ax_h1.grid(alpha=0.25)

ax_h2 = fig.add_subplot(gs[1, 2:4])
ax_h2.hist(norm_m1_p1.ravel(), bins=64, range=(-1.0, 1.0), color="#55a868", alpha=0.85)
ax_h2.set_title("Distribusi Piksel Masuk MobileNetV2 (float32: -1.0 s/d +1.0)")
ax_h2.set_xlabel("Nilai Ternormalisasi (pixel / 127.5 - 1.0)")
ax_h2.set_ylabel("Frekuensi")
ax_h2.grid(alpha=0.25)

fig.suptitle(f"Tahap 1: Preprocessing Proyek FERQUEUE — {active_name}", fontsize=14, fontweight="bold")
save_fig(fig, "01_preprocessing_ferqueue.png")
plt.show()

print(f"Contoh nilai piksel [0,0] RGB mentah      : {resized_320[0, 0].tolist()}")
print(f"Contoh nilai piksel [0,0] ternormalisasi  : {np.round(norm_m1_p1[0, 0], 4).tolist()}")

## Cell 4 — Tahap 2: Membedah Ekstraksi Fitur Otomatis (MobileNetV2 + FPNLite P3–P7)

Walaupun ekstraksi fitur di CNN berjalan otomatis di dalam model, kita bisa **mengekstrak langsung tensor *Feature Maps* intermediate** dari `SavedModel` hasil trainingmu sendiri menggunakan *graph pruning* TensorFlow:
* **Layer Awal MobileNetV2 (`160×160×32` & `80×80×96`)**: mengekstrak fitur *low-level* seperti tepi, garis bodi, jendela, dan tekstur roda.
* **FPNLite Piramida Fitur (`P3` s/d `P7`, masing-masing 128 channel)**:
  - **P3 (`40×40×128`)** → resolusi tinggi untuk mendeteksi kendaraan kecil / jauh.
  - **P4 (`20×20×128`) & P5 (`10×10×128`)** → kendaraan berukuran sedang.
  - **P6 (`5×5×128`) & P7 (`3×3×128`)** → fitur semantik global untuk kendaraan besar (`Truk` / `Bus`).
* **Activation Heatmap Overlay**: menumpuk rata-rata energi aktivasi FPN di atas gambar asli untuk melihat **area mana pada kendaraan yang paling diperhatikan model**.

In [ ]:
# Bangun sub-graph ekstraktor feature map langsung dari SavedModel FERQUEUE
print("Mengekstrak graph internal MobileNetV2 + FPNLite dari SavedModel (tunggu ~5-15 detik)...")
frozen_fn = frozen_fn if "frozen_fn" in globals() else convert_variables_to_constants_v2(detector_fn, lower_control_flow=False)
all_tensor_names = [out.name for op in frozen_fn.graph.get_operations() for out in op.outputs]

LAYER_PATTERNS = [
    ("Low-Level (160x160x32)", "expanded_conv_depthwise_relu/Relu6:0"),
    ("Mid-Level (80x80x96)", "block_1_depthwise_relu/Relu6:0"),
    ("FPN P3 (40x40x128)", "PredictionTower/conv2d_3/activation_0/Relu6:0"),
    ("FPN P4 (20x20x128)", "PredictionTower/conv2d_3/activation_1/Relu6:0"),
    ("FPN P5 (10x10x128)", "PredictionTower/conv2d_3/activation_2/Relu6:0"),
    ("FPN P6 (5x5x128)", "PredictionTower/conv2d_3/activation_3/Relu6:0"),
    ("FPN P7 (3x3x128)", "PredictionTower/conv2d_3/activation_4/Relu6:0"),
]

resolved_layers = []
for label, suffix in LAYER_PATTERNS:
    match = next((name for name in all_tensor_names if name.endswith(suffix)), None)
    if match:
        resolved_layers.append((label, match))

if not resolved_layers:
    raise RuntimeError("Tensor feature map MobileNetV2/FPNLite tidak ditemukan pada SavedModel ini.")

fetch_tensors = [frozen_fn.graph.get_tensor_by_name(name) for _, name in resolved_layers]
feature_extractor_fn = frozen_fn.prune(frozen_fn.inputs, fetch_tensors)

input_batch = tf.convert_to_tensor(proc_rgb[None, ...], dtype=tf.uint8)
raw_fmaps = feature_extractor_fn(input_batch)
feature_maps = {label: t.numpy()[0] for (label, _), t in zip(resolved_layers, raw_fmaps)}

# Tabel ringkasan statistik fitur tiap layer
stats_rows = []
for label, fmap in feature_maps.items():
    h_f, w_f, c_f = fmap.shape
    stats_rows.append({
        "Tahap_Layer": label,
        "Resolusi_Spasial": f"{h_f}x{w_f}",
        "Jumlah_Channel": c_f,
        "Total_Nilai_Fitur": h_f * w_f * c_f,
        "Aktivasi_Mean": round(float(fmap.mean()), 4),
        "Aktivasi_Max": round(float(fmap.max()), 4),
    })
df_fmap_stats = pd.DataFrame(stats_rows)
display(df_fmap_stats)

# Plot 1: Visualisasi Feature Maps (Mean aktivasi & 3 channel terkuat per tahap utama)
selected_for_grid = [k for k in feature_maps.keys() if not k.startswith("FPN P7")]
fig, axes = plt.subplots(len(selected_for_grid), 5, figsize=(16, 2.7 * len(selected_for_grid)))

for r, label in enumerate(selected_for_grid):
    fmap = feature_maps[label]  # shape: (H, W, C)
    mean_map = fmap.mean(axis=-1)
    ch_energy = fmap.mean(axis=(0, 1))
    top_channels = np.argsort(-ch_energy)[:4]

    axes[r, 0].imshow(mean_map, cmap="viridis")
    axes[r, 0].set_title(f"{label}\nMean Semua Channel", fontsize=9)
    for c_idx, ch in enumerate(top_channels, start=1):
        axes[r, c_idx].imshow(fmap[:, :, ch], cmap="magma")
        axes[r, c_idx].set_title(f"Channel #{ch}\n(mean={ch_energy[ch]:.2f})", fontsize=8.5)
    for c in range(5):
        axes[r, c].axis("off")

fig.suptitle(f"Tahap 2: Ekstraksi Fitur Otomatis MobileNetV2 + FPNLite — {active_name}", fontsize=14, fontweight="bold")
plt.tight_layout()
save_fig(fig, "02_feature_maps_mobilenet.png")
plt.show()


# Plot 2: Activation Heatmap Overlay (menggabungkan skala P3, P4, P5 ke resolusi asli)
def build_fpn_heatmap(fmaps_dict: dict, target_hw: tuple) -> np.ndarray:
    H, W = target_hw
    acc = np.zeros((H, W), dtype=np.float32)
    used = 0
    for key in ("FPN P3 (40x40x128)", "FPN P4 (20x20x128)", "FPN P5 (10x10x128)"):
        if key in fmaps_dict:
            m = fmaps_dict[key].mean(axis=-1)
            m = (m - m.min()) / (m.max() - m.min() + 1e-6)
            acc += cv2.resize(m, (W, H), interpolation=cv2.INTER_CUBIC)
            used += 1
    acc /= max(1, used)
    acc = (acc - acc.min()) / (acc.max() - acc.min() + 1e-6)
    return acc


heatmap_norm = build_fpn_heatmap(feature_maps, proc_rgb.shape[:2])
heatmap_color = cv2.cvtColor(cv2.applyColorMap((heatmap_norm * 255).astype(np.uint8), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
heatmap_overlay = cv2.addWeighted(proc_rgb, 0.55, heatmap_color, 0.45, 0)

fig2, axes2 = plt.subplots(1, 3, figsize=(15, 4.8))
axes2[0].imshow(proc_rgb)
axes2[0].set_title("1. Citra Masukan")
axes2[1].imshow(heatmap_norm, cmap="jet")
axes2[1].set_title("2. Peta Aktivasi FPNLite (P3+P4+P5)")
axes2[2].imshow(heatmap_overlay)
axes2[2].set_title("3. Overlay Fokus Perhatian MobileNetV2")
for ax in axes2:
    ax.axis("off")
fig2.suptitle(f"Visualisasi Fokus Ekstraksi Fitur CNN — {active_name}", fontsize=13.5, fontweight="bold")
plt.tight_layout()
save_fig(fig2, "03_activation_heatmap.png")
plt.show()

## Cell 5 — Tahap 3: Deteksi & Post-Processing (Raw Anchors → Confidence Threshold → NMS)

Mengikuti `apps/computer-vision/src/ferqueue_cv/postprocessing.py` dan `receiver-v3.json`:
1. **Raw Predictions**: SSD menghasilkan **12.804 kandidat anchor** dari piramida fitur P3–P7 (kita tampilkan 40 kandidat teratas sebelum disaring).
2. **Confidence Threshold (`>= 0.50`)**: membuang kandidat latar belakang dengan skor rendah.
3. **Class-Agnostic NMS (`IoU <= 0.45`)**: menghapus kotak duplikat yang saling tumpang tindih pada objek yang sama hingga menyisakan **Bounding Box akhir + Label Kelas + Confidence Score**.

In [ ]:
CONFIDENCE_THRESHOLD = 0.50
NMS_IOU_THRESHOLD = 0.45
MAX_DETECTIONS = 10
CLASS_AGNOSTIC_NMS = True

CLASS_COLORS = {
    "Mobil": (40, 210, 90),
    "Truk": (255, 165, 0),
    "Bus": (65, 155, 255),
}


def select_detections_ferqueue(result: dict, conf_th=0.5, iou_th=0.45, max_det=10, class_agnostic=True):
    """Persis mengikuti apps/computer-vision/src/ferqueue_cv/postprocessing.py."""
    boxes = np.asarray(result["detection_boxes"])
    scores = np.asarray(result["detection_scores"])
    classes = np.asarray(result["detection_classes"])
    candidates = [
        int(idx)
        for idx in np.argsort(-scores, kind="stable")
        if scores[idx] >= conf_th
        and np.isfinite(scores[idx])
        and np.isfinite(boxes[idx]).all()
        and np.isfinite(classes[idx])
        and boxes[idx][2] > boxes[idx][0]
        and boxes[idx][3] > boxes[idx][1]
    ]
    keep = []
    while candidates and len(keep) < max_det:
        current = candidates.pop(0)
        keep.append(current)
        remaining = []
        a = boxes[current]
        for idx in candidates:
            if not class_agnostic and classes[idx] != classes[current]:
                remaining.append(idx)
                continue
            b = boxes[idx]
            inter = max(0.0, min(a[2], b[2]) - max(a[0], b[0])) * max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
            union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
            if union <= 0 or inter / union <= iou_th:
                remaining.append(idx)
        candidates = remaining
    return keep


def run_ssd_inference(rgb: np.ndarray):
    t_start = perf_counter()
    raw_out = detector_fn(tf.convert_to_tensor(rgb[None, ...], dtype=tf.uint8))
    elapsed_ms = (perf_counter() - t_start) * 1000.0
    count = int(raw_out["num_detections"][0])
    parsed = {
        "detection_boxes": raw_out["detection_boxes"][0, :count].numpy(),
        "detection_scores": raw_out["detection_scores"][0, :count].numpy(),
        "detection_classes": raw_out["detection_classes"][0, :count].numpy(),
        "raw_detection_boxes": raw_out["raw_detection_boxes"][0].numpy(),
        "raw_detection_scores": raw_out["raw_detection_scores"][0].numpy(),
        "inference_ms": elapsed_ms,
    }
    return parsed


# Warmup 1x agar catatan waktu inferensi akurat
_ = detector_fn(tf.convert_to_tensor(proc_rgb[None, ...], dtype=tf.uint8))
ssd_out = run_ssd_inference(proc_rgb)

H, W = proc_rgb.shape[:2]

# Panel 1: Top-40 raw anchor candidates (sebelum NMS internal/eksternal)
raw_scores_fg = ssd_out["raw_detection_scores"][:, 1:].max(axis=-1)
top40_idx = np.argsort(-raw_scores_fg)[:40]
vis_raw = proc_rgb.copy()
for i in top40_idx:
    y1, x1, y2, x2 = ssd_out["raw_detection_boxes"][i]
    cv2.rectangle(vis_raw, (int(x1 * W), int(y1 * H)), (int(x2 * W), int(y2 * H)), (255, 80, 80), 1)

# Panel 2: Lolos Confidence Threshold (>= 0.5) sebelum NMS proyek
vis_pre_nms = proc_rgb.copy()
pre_nms_idx = [i for i, s in enumerate(ssd_out["detection_scores"]) if s >= CONFIDENCE_THRESHOLD]
for i in pre_nms_idx:
    y1, x1, y2, x2 = ssd_out["detection_boxes"][i]
    cv2.rectangle(vis_pre_nms, (int(x1 * W), int(y1 * H)), (int(x2 * W), int(y2 * H)), (255, 215, 0), 2)

# Panel 3: Hasil Akhir setelah Confidence + Class-Agnostic NMS
kept_idx = select_detections_ferqueue(
    ssd_out, CONFIDENCE_THRESHOLD, NMS_IOU_THRESHOLD, MAX_DETECTIONS, CLASS_AGNOSTIC_NMS
)
vis_final = proc_rgb.copy()
final_detections = []
for i in kept_idx:
    y1, x1, y2, x2 = map(float, ssd_out["detection_boxes"][i])
    score = float(ssd_out["detection_scores"][i])
    cls_id = int(ssd_out["detection_classes"][i])
    cls_name = CLASSES[cls_id - 1] if 1 <= cls_id <= len(CLASSES) else f"ID-{cls_id}"
    color = CLASS_COLORS.get(cls_name, (40, 210, 90))
    px1, py1 = max(0, int(x1 * W)), max(0, int(y1 * H))
    px2, py2 = min(W, int(x2 * W)), min(H, int(y2 * H))
    cv2.rectangle(vis_final, (px1, py1), (px2, py2), color, 3)
    label_txt = f"{cls_name} {score:.2f}"
    (tw, th), _ = cv2.getTextSize(label_txt, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
    cv2.rectangle(vis_final, (px1, max(0, py1 - th - 10)), (px1 + tw + 8, max(th + 10, py1)), color, -1)
    cv2.putText(vis_final, label_txt, (px1 + 4, max(th + 4, py1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2, cv2.LINE_AA)
    final_detections.append({
        "Kelas": cls_name,
        "Confidence": round(score, 4),
        "box_norm": {"ymin": y1, "xmin": x1, "ymax": y2, "xmax": x2},
        "BBox_px": f"({px1}, {py1}, {px2 - px1}x{py2 - py1})",
        "Inferensi_ms": round(ssd_out["inference_ms"], 1),
    })

fig, axes = plt.subplots(1, 3, figsize=(16.5, 5.2))
axes[0].imshow(vis_raw)
axes[0].set_title(f"1. Raw Anchor Candidates\n(Top 40 dari {len(raw_scores_fg):,} anchor)")
axes[1].imshow(vis_pre_nms)
axes[1].set_title(f"2. Setelah Confidence >= {CONFIDENCE_THRESHOLD}\n({len(pre_nms_idx)} kandidat sebelum NMS)")
axes[2].imshow(vis_final)
axes[2].set_title(f"3. Hasil Akhir Setelah NMS (IoU<={NMS_IOU_THRESHOLD})\n({len(final_detections)} deteksi | {ssd_out['inference_ms']:.1f} ms)")
for ax in axes:
    ax.axis("off")

fig.suptitle(f"Tahap 3: Deteksi & Post-Processing SSD MobileNetV2 — {active_name}", fontsize=14, fontweight="bold")
plt.tight_layout()
save_fig(fig, "04_raw_vs_nms_deteksi.png")
plt.show()

if final_detections:
    df_det = pd.DataFrame([{k: v for k, v in d.items() if k != "box_norm"} for d in final_detections])
    display(df_det)
else:
    print("Tidak ada objek yang melewati CONFIDENCE_THRESHOLD. Coba turunkan nilainya (misal 0.30).")

## Cell 6 — Tahap 4: Segmentasi Pasca-Deteksi (GrabCut + Morfologi sesuai `processing.py`)

Sesuai dokumentasi proyek FERQUEUE (`docs/PCV_MANUAL.md` & `ferqueue_cv/processing.py`):
* Model SSD MobileNetV2 **tidak memerlukan segmentasi piksel sebelum deteksi** (gambar utuh langsung masuk ke jaringan).
* Namun setelah **Bounding Box** diperoleh dari SSD, proyek ini menyediakan **GrabCut + Operasi Morfologi (Opening & Closing)** untuk memisahkan siluet *foreground* kendaraan dari *background* (sebagai tahap review / analisis bentuk objek).

In [ ]:
GRABCUT_ITERATIONS = 5
GRABCUT_MORPH_KERNEL = 3


def grabcut_mask_ferqueue(rgb: np.ndarray, box_norm: dict, iterations=5, morph_k=3):
    """Persis mengikuti grabcut_mask() di apps/computer-vision/src/ferqueue_cv/processing.py."""
    height, width = rgb.shape[:2]
    x1 = max(0, math.floor(box_norm["xmin"] * width))
    y1 = max(0, math.floor(box_norm["ymin"] * height))
    x2 = min(width, math.ceil(box_norm["xmax"] * width))
    y2 = min(height, math.ceil(box_norm["ymax"] * height))
    if x2 - x1 < 2 or y2 - y1 < 2:
        return None, "box_too_small"
    padded = cv2.copyMakeBorder(rgb, 1, 1, 1, 1, cv2.BORDER_REPLICATE)
    mask = np.zeros(padded.shape[:2], dtype=np.uint8)
    cv2.setRNGSeed(42)
    try:
        cv2.grabCut(
            padded, mask, (x1 + 1, y1 + 1, x2 - x1, y2 - y1),
            np.zeros((1, 65)), np.zeros((1, 65)), iterations, cv2.GC_INIT_WITH_RECT,
        )
    except cv2.error:
        return None, "grabcut_failed"
    fg = np.isin(mask[1:-1, 1:-1], (cv2.GC_FGD, cv2.GC_PR_FGD)).astype(np.uint8) * 255
    if morph_k:
        elem = np.ones((morph_k, morph_k), dtype=np.uint8)
        fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, elem)
        fg = cv2.morphologyEx(fg, cv2.MORPH_CLOSE, elem)
    bounded = np.zeros_like(fg)
    bounded[y1:y2, x1:x2] = fg[y1:y2, x1:x2]
    if not bounded.any():
        return None, "empty_mask"
    return bounded, "ok"


combined_grabcut = np.zeros(proc_rgb.shape[:2], dtype=np.uint8)
for det in final_detections:
    m, status = grabcut_mask_ferqueue(proc_rgb, det["box_norm"], GRABCUT_ITERATIONS, GRABCUT_MORPH_KERNEL)
    if m is not None:
        combined_grabcut = cv2.bitwise_or(combined_grabcut, m)

cutout_gc = cv2.bitwise_and(proc_rgb, proc_rgb, mask=combined_grabcut)
overlay_gc = proc_rgb.copy().astype(np.float32)
tint = np.zeros_like(overlay_gc)
tint[:, :] = (255, 70, 20)
m_bool = combined_grabcut > 0
overlay_gc[m_bool] = 0.55 * overlay_gc[m_bool] + 0.45 * tint[m_bool]
overlay_gc = overlay_gc.astype(np.uint8)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
axes[0].imshow(vis_final)
axes[0].set_title("1. ROI dari Bbox SSD")
axes[1].imshow(combined_grabcut, cmap="gray")
axes[1].set_title("2. Mask GrabCut + Morfologi")
axes[2].imshow(cutout_gc)
axes[2].set_title("3. Cutout Foreground")
axes[3].imshow(overlay_gc)
axes[3].set_title("4. Overlay Segmentasi")
for ax in axes:
    ax.axis("off")

fig.suptitle(f"Tahap 4: Segmentasi Pasca-Deteksi (GrabCut Dipandu Bbox SSD) — {active_name}", fontsize=13.5, fontweight="bold")
plt.tight_layout()
save_fig(fig, "05_segmentasi_grabcut.png")
plt.show()

## Cell 7 — Ringkasan Alur 1-Baris Proyek FERQUEUE (Siap PPT!) & Multi-Gambar

Membuat 1 gambar ringkasan horizontal 5-panel dari awal sampai akhir yang langsung siap ditempel di PowerPoint, lalu menjalankan deteksi pada semua gambar uji yang kamu muat.

In [ ]:
# 1. Ringkasan Alur 1-Baris (5 Panel)
fig, axes = plt.subplots(1, 5, figsize=(20, 4.3))
summary_panels = [
    ("1. Input Citra\n(RGB Asli)", proc_rgb),
    ("2. Preprocessing\n(Resize 320x320, [-1,1])", resized_320),
    ("3. Ekstraksi Fitur\n(MobileNetV2+FPNLite)", heatmap_overlay),
    ("4. Hasil Deteksi\n(Bbox + Label + NMS)", vis_final),
    ("5. Segmentasi ROI\n(GrabCut + Morfologi)", overlay_gc),
]
for ax, (title, im) in zip(axes, summary_panels):
    ax.imshow(im)
    ax.set_title(title, fontsize=10.5)
    ax.axis("off")

fig.suptitle(f"Ringkasan Alur Proyek FERQUEUE (SSD MobileNetV2 FPNLite) — {active_name}", fontsize=14, fontweight="bold")
plt.tight_layout()
save_fig(fig, "06_alur_lengkap_ferqueue.png")
plt.show()

# 2. Evaluasi pada semua gambar uji yang dimuat
if len(loaded_images) > 1:
    cols = min(len(loaded_images), 3)
    rows = math.ceil(len(loaded_images) / cols)
    fig_m, axes_m = plt.subplots(rows, cols, figsize=(5.2 * cols, 4.2 * rows))
    axes_m = np.atleast_1d(axes_m).ravel()
    for idx, ax in enumerate(axes_m):
        if idx < len(loaded_images):
            nm, pil_i = loaded_images[idx]
            arr_i = np.asarray(preprocess_image_ferqueue(pil_i, MODEL_PREPROCESSING), dtype=np.uint8)
            out_i = run_ssd_inference(arr_i)
            keep_i = select_detections_ferqueue(out_i, CONFIDENCE_THRESHOLD, NMS_IOU_THRESHOLD, MAX_DETECTIONS, CLASS_AGNOSTIC_NMS)
            vis_i = arr_i.copy()
            hi, wi = arr_i.shape[:2]
            labels_found = []
            for k in keep_i:
                y1, x1, y2, x2 = map(float, out_i["detection_boxes"][k])
                sc = float(out_i["detection_scores"][k])
                cid = int(out_i["detection_classes"][k])
                cname = CLASSES[cid - 1] if 1 <= cid <= len(CLASSES) else str(cid)
                col_c = CLASS_COLORS.get(cname, (40, 210, 90))
                cv2.rectangle(vis_i, (int(x1 * wi), int(y1 * hi)), (int(x2 * wi), int(y2 * hi)), col_c, 3)
                cv2.putText(vis_i, f"{cname} {sc:.2f}", (int(x1 * wi) + 4, max(24, int(y1 * hi) + 22)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, col_c, 2, cv2.LINE_AA)
                labels_found.append(f"{cname}({sc:.2f})")
            ax.imshow(vis_i)
            ax.set_title(f"{nm[:22]}\n{', '.join(labels_found) or 'Tidak terdeteksi'} ({out_i['inference_ms']:.0f}ms)", fontsize=9)
        ax.axis("off")
    fig_m.suptitle("Hasil Deteksi SSD MobileNetV2 pada Semua Gambar Uji", fontsize=13.5, fontweight="bold")
    plt.tight_layout()
    save_fig(fig_m, "07_multi_gambar_ferqueue.png")
    plt.show()

## Cell 8 — *(Opsional)* Modul Kalibrasi Koin 50 Yen (`side_measurement.py`) & Unduh ZIP Output PPT

Di proyek FERQUEUE juga terdapat modul `apps/computer-vision/src/ferqueue_cv/side_measurement.py` yang memakai **PCV Klasik (Grayscale → Gaussian Blur → Otsu Thresholding → Contour → Ellipse Fitting)** pada koin referensi 50 yen (`diameter = 21.0 mm`) untuk mengestimasi panjang dan tinggi fisik kendaraan (dalam `cm`).

* Ubah `RUN_COIN_DEMO = True` dan atur `COIN_REGION` (`ymin, xmin, ymax, xmax` ternormalisasi `0..1`) jika foto ujimu memiliki koin 50 yen.
* Di bagian bawah, semua gambar PNG (300 DPI) dikemas menjadi `nb2_ferqueue_mobilenet_outputs.zip`.

In [ ]:
RUN_COIN_DEMO = False
COIN_REGION = (0.70, 0.05, 0.98, 0.30)  # (ymin, xmin, ymax, xmax) area koin pada gambar
COIN_DIAMETER_MM = 21.0

if RUN_COIN_DEMO:
    y1c, x1c, y2c, x2c = COIN_REGION
    left, top = int(x1c * W), int(y1c * H)
    right, bottom = math.ceil(x2c * W), math.ceil(y2c * H)
    crop_gray = cv2.cvtColor(proc_rgb[top:bottom, left:right], cv2.COLOR_RGB2GRAY)
    crop_h, crop_w = crop_gray.shape[:2]
    blur_c = cv2.GaussianBlur(crop_gray, (5, 5), 0)
    _, otsu_c = cv2.threshold(blur_c, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
    candidates = []
    if min(crop_w, crop_h) >= 28:
        for binary in (otsu_c, cv2.bitwise_not(otsu_c)):
            cnts_c, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            for cnt in cnts_c:
                _, _, wc, hc = cv2.boundingRect(cnt)
                if len(cnt) < 5 or min(wc, hc) < 24:
                    continue
                (cx, cy), axes_e, _ = cv2.fitEllipse(cnt)
                minor, major = sorted(axes_e)
                if not (24 <= minor <= major and minor / major >= 0.88):
                    continue
                margin = major / 2.0 + 2.0
                if not (margin <= cx <= crop_w - margin and margin <= cy <= crop_h - margin):
                    continue
                if major < 0.4 * min(crop_w, crop_h):
                    continue
                fill = cv2.contourArea(cnt) / (math.pi * minor * major / 4.0)
                if not 0.82 <= fill <= 1.12:
                    continue
                diam = major + 1.0
                candidates.append({"cx": cx, "cy": cy, "diameter_px": diam, "mm_per_px": COIN_DIAMETER_MM / diam})
    coin_found = candidates[0] if len(candidates) == 1 else None

    fig_c, axes_c = plt.subplots(1, 3, figsize=(12, 3.8))
    axes_c[0].imshow(crop_gray, cmap="gray")
    axes_c[0].set_title("1. Crop ROI Koin (Grayscale)")
    axes_c[1].imshow(otsu_c, cmap="gray")
    axes_c[1].set_title("2. Segmentasi Otsu Threshold")
    vis_coin = cv2.cvtColor(crop_gray, cv2.COLOR_GRAY2RGB)
    if coin_found:
        cv2.circle(vis_coin, (int(coin_found["cx"]), int(coin_found["cy"])), int(coin_found["diameter_px"] / 2), (40, 220, 80), 2)
        axes_c[2].set_title(f"3. Koin Terdeteksi\nd={coin_found['diameter_px']:.1f}px ({coin_found['mm_per_px']:.3f} mm/px)")
        if len(final_detections) == 1:
            b = final_detections[0]["box_norm"]
            cm_per_px = coin_found["mm_per_px"] / 10.0
            L_cm = (b["xmax"] - b["xmin"]) * W * cm_per_px
            H_cm = (b["ymax"] - b["ymin"]) * H * cm_per_px
            print(f"Estimasi Dimensi Samping Kendaraan: Panjang={L_cm:.2f} cm | Tinggi={H_cm:.2f} cm")
    else:
        axes_c[2].set_title("3. Koin Tidak Terdeteksi di ROI")
    axes_c[2].imshow(vis_coin)
    for ax in axes_c:
        ax.axis("off")
    plt.tight_layout()
    save_fig(fig_c, "08_kalibrasi_koin_otsu.png")
    plt.show()

zip_path = OUT_DIR.parent / "nb2_ferqueue_mobilenet_outputs.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT_DIR.glob("*")):
        if p.is_file():
            zf.write(p, p.name)
print(f"Semua gambar PPT tersimpan di: {zip_path} ({zip_path.stat().st_size / 1024:.1f} KB)")
# Hapus tanda '#' di bawah jika ingin langsung mengunduh ZIP di Colab:
# if IN_COLAB:
#     colab_files.download(str(zip_path))